In [45]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score , precision_score  , recall_score , f1_score , confusion_matrix
from xgboost import XGBClassifier



df = pd.read_csv("D:/AYUVA/ml/data/processed/cardio_clean3.csv")



features = ['age_year', 'ap_hi', 'ap_lo', 
            'cholesterol', 'gluc', 'smoke', 'alco', 'active','bmi','pulse_pressure','map']

target = "cardio"

x = df[features]
y = df[target]

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size = 0.2,random_state = 42 , stratify= y )
#we dont need scaling for xgboost

In [46]:
model = XGBClassifier()
train_fit = model.fit(x_train,y_train)
x_test_predict = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, x_test_predict))
print("Precision:", precision_score(y_test, x_test_predict))
print("Recall:", recall_score(y_test, x_test_predict))
print("F1:", f1_score(y_test, x_test_predict))
print("Confusion Matrix:\n", confusion_matrix(y_test, x_test_predict))

Accuracy: 0.7292142857142857
Precision: 0.7473375520913721
Recall: 0.6921097770154374
F1: 0.7186641929499072
Confusion Matrix:
 [[5367 1637]
 [2154 4842]]


In [47]:
param_grid = {
    'n_estimators' : [100,200,300],
    'max_depth' : [3,5,7],
    'learning_rate': [0.01,0.1,0.2,0.3],
    'subsample':[0.6,0.7,0.8,1.0]              #every tree will we trained of 80% of random data means each tree is differently trained 
}

grid_search = GridSearchCV(
    XGBClassifier(),
    param_grid,
    cv = 3 ,                                  #3-fold-cross validation
    scoring = 'recall',                          #priority -- recall 
    n_jobs = -1                                   #use all cores 
)

grid_search.fit(x_train,y_train)
print("Best params:", grid_search.best_params_)
print("Best CV recall:", grid_search.best_score_)


Best params: {'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV recall: 0.7001750314765784


In [48]:
best_model = XGBClassifier(**grid_search.best_params_)
best_model.fit(x_train,y_train)
final_predict = best_model.predict(x_test)


print("Final Accuracy:", accuracy_score(y_test, final_predict))
print("Final Precision:", precision_score(y_test, final_predict))
print("Final Recall:", recall_score(y_test, final_predict))
print("Final F1:", f1_score(y_test, final_predict))
print("Confusion Matrix:\n", confusion_matrix(y_test, final_predict))

Final Accuracy: 0.7336428571428572
Final Precision: 0.7511917576503152
Final Recall: 0.698256146369354
Final F1: 0.7237573153566931
Confusion Matrix:
 [[5386 1618]
 [2111 4885]]
